# Experimento (rama `experimento-calibracion-duracion`) - tolerancia de pausa
calibrada contra duraciones reales anotadas

**Excepción deliberada y acotada a la regla "sin motor viejo"**: se exporta
**solo** la distribución de duraciones (`t_fin - t_inicio`) de las 743
anotaciones reales de `Ciclo_Alpha_v2/fase_0_ruido/data/anotaciones_av2.csv`
— nada de features, nada de modelo, nada de predicciones. Se usa únicamente
como **objetivo de calibración** para el único parámetro libre que nos
queda: cuánta pausa tolerar antes de cortar un segmento activo.

Las anotaciones son 100% KPCL0034 (comedero) — no hay ninguna de KPCL0035.
El τ elegido se prueba igual en KPCL0035, como experimento fuera de dominio.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
ANOTACIONES_CSV = NOTEBOOK_DIR.parent / "Ciclo_Alpha_v2" / "fase_0_ruido" / "data" / "anotaciones_av2.csv"

anotaciones = pd.read_csv(ANOTACIONES_CSV)
anotaciones["t_inicio"] = pd.to_datetime(anotaciones["t_inicio"], format="ISO8601", utc=True)
anotaciones["t_fin"] = pd.to_datetime(anotaciones["t_fin"], format="ISO8601", utc=True)
anotaciones["duracion_s"] = (anotaciones["t_fin"] - anotaciones["t_inicio"]).dt.total_seconds()

print(anotaciones["categoria"].value_counts())
print(anotaciones.groupby("categoria")["duracion_s"].describe().round(1))

# --- Exportar SOLO percentiles de duracion (el objetivo de calibracion) --------
PCTS = [10, 25, 50, 75, 90, 95, 99]
filas_ref = []
for _cat, _g in anotaciones.groupby("categoria"):
    _fila = {"categoria": _cat, "n": len(_g)}
    _fila.update({
        f"p{p}_duracion_s": round(_g["duracion_s"].quantile(p / 100), 1) for p in PCTS
    })
    filas_ref.append(_fila)
_fila_todas = {"categoria": "TODAS", "n": len(anotaciones)}
_fila_todas.update({
    f"p{p}_duracion_s": round(anotaciones["duracion_s"].quantile(p / 100), 1) for p in PCTS
})
filas_ref.append(_fila_todas)
duraciones_ref_df = pd.DataFrame(filas_ref).set_index("categoria")

REF_CSV = NOTEBOOK_DIR / "data" / "duraciones_referencia_anotaciones.csv"
duraciones_ref_df.to_csv(REF_CSV)
print(f"Exportado: {REF_CSV}")
duraciones_ref_df


## Barrido de τ (tolerancia de pausa, en segundos) contra el objetivo real

Se reemplaza el Hallazgo 6 original ("tolera 1 lectura", ≈30s fijo) por un
corte basado en **tiempo acumulado** de `paso_estable` consecutivo: un corte
real recién ocurre si la racha estable acumula ≥ τ segundos (o es un gap real,
o hay un NaN). Se prueban varios τ y se compara la duración resultante de los
candidatos contra el objetivo real (`TODAS`, arriba) — el objetivo mezcla las
3 categorías porque nuestros candidatos tampoco están separados por categoría
todavía.


In [ ]:
CACHE_CSV = NOTEBOOK_DIR / "data" / "lecturas_limpias.csv"
GAP_CUTOFF_S = 300

df = pd.read_csv(CACHE_CSV)
df["device_id"] = df["device_id"].astype("category")
df["device_code"] = df["device_code"].astype("category")
df["ts"] = pd.to_datetime(df["ts"], format="ISO8601", utc=True)
df["delta_peso"] = df.groupby("device_id", observed=True)["peso"].diff()
df["delta_t"] = df.groupby("device_id", observed=True)["ts"].diff().dt.total_seconds()
df["abs_delta_peso"] = df["delta_peso"].abs()
is_gap = df["delta_t"] > GAP_CUTOFF_S
paso_estable = (df["delta_peso"] == 0) & (~is_gap.fillna(False))
DEVICE_CODES = df["device_code"].cat.categories

def construir_candidatos(tau_s):
    """Igual logica que el Hallazgo 6, pero el corte exige tau_s SEGUNDOS
    acumulados de paso_estable consecutivo, no una cantidad fija de lecturas."""
    _cambio = (paso_estable != paso_estable.shift(1)) | (df["device_code"] != df["device_code"].shift(1))
    _racha_id = _cambio.cumsum()
    _racha_dur = df.groupby(_racha_id)["delta_t"].transform("sum")
    _es_corte_real = is_gap.fillna(False) | df["delta_peso"].isna() | (paso_estable & (_racha_dur >= tau_s))
    _es_movimiento = ~_es_corte_real
    _cid = _es_corte_real.groupby(df["device_code"], observed=True).cumsum()
    _filas = []
    for _code in DEVICE_CODES:
        _mask = _es_movimiento & (df["device_code"] == _code)
        _grp = df.loc[_mask].groupby(_cid[_mask], observed=True)
        _seg = pd.DataFrame({
            "device_code": _code, "n_lecturas": _grp.size(), "duracion_s": _grp["delta_t"].sum(),
            "delta_neto_g": _grp["delta_peso"].sum(), "max_abs_delta_g": _grp["abs_delta_peso"].max(),
            "idx_inicio": _grp.apply(lambda g: g.index.min()), "idx_fin": _grp.apply(lambda g: g.index.max()),
        })
        _filas.append(_seg)
    _segmentos = pd.concat(_filas, ignore_index=True)
    _segmentos["es_candidato"] = False
    for _code, _g in _segmentos.groupby("device_code", observed=True):
        _mediana = _g["max_abs_delta_g"].median()
        _mad = (_g["max_abs_delta_g"] - _mediana).abs().median()
        _umbral = _mediana + (3.5 / 0.6745) * _mad
        _segmentos.loc[_g.index, "es_candidato"] = _g["max_abs_delta_g"] > _umbral
    return _segmentos[_segmentos["es_candidato"]].copy()

_objetivo = duraciones_ref_df.loc["TODAS"]
print(f"Objetivo real (TODAS las categorias): P50={_objetivo['p50_duracion_s']:.0f}s  P75={_objetivo['p75_duracion_s']:.0f}s  P90={_objetivo['p90_duracion_s']:.0f}s")
print()
for _tau in (30, 60, 90, 120, 180, 240, 300):
    _cand = construir_candidatos(_tau)
    _k34 = _cand[_cand["device_code"] == "KPCL0034"]
    _p = _k34["duracion_s"].quantile([.5, .75, .9])
    print(f"tau={_tau:>3}s -> n={len(_k34):,}  P50={_p[.5]:.0f}s  P75={_p[.75]:.0f}s  P90={_p[.9]:.0f}s")


## Lectura del barrido

Pendiente de completar con el resultado real una vez corrido — elegir el τ
que más se acerca al objetivo, documentando explícitamente si lo alcanza
del todo o queda una brecha (y de cuánto).


## Reconstrucción final con el τ elegido + clustering (mismo esquema para la app)


In [ ]:
TAU_ELEGIDO_S = 180  # ver celda de barrido y lectura de arriba

candidatos_df = construir_candidatos(TAU_ELEGIDO_S).reset_index(drop=True)
K_MARGEN = 5
niveles_antes, niveles_despues = [], []
for _code, _sub in candidatos_df.groupby("device_code", observed=True):
    _idx_estable = np.array(sorted(df.index[(df["device_code"] == _code) & paso_estable].tolist()))
    for _, _row in _sub.iterrows():
        _pi = np.searchsorted(_idx_estable, _row["idx_inicio"])
        _antes = _idx_estable[max(0, _pi - K_MARGEN):_pi]
        _pf = np.searchsorted(_idx_estable, _row["idx_fin"], side="right")
        _despues = _idx_estable[_pf:_pf + K_MARGEN]
        niveles_antes.append(df.loc[_antes, "peso"].median() if len(_antes) == K_MARGEN else np.nan)
        niveles_despues.append(df.loc[_despues, "peso"].median() if len(_despues) == K_MARGEN else np.nan)
candidatos_df["nivel_antes"] = niveles_antes
candidatos_df["nivel_despues"] = niveles_despues
candidatos_df["delta_neto_real"] = candidatos_df["nivel_despues"] - candidatos_df["nivel_antes"]
candidatos_df["ts_inicio"] = df.loc[candidatos_df["idx_inicio"], "ts"].reset_index(drop=True)
candidatos_df["ts_fin"] = df.loc[candidatos_df["idx_fin"], "ts"].reset_index(drop=True)
candidatos_df = candidatos_df.dropna(subset=["delta_neto_real"])
candidatos_df["candidato_id"] = (
    candidatos_df["device_code"].astype(str) + "_" +
    candidatos_df["ts_inicio"].dt.strftime("%Y%m%d%H%M%S%f")
)

# n_cambios_signo -- recalcular sobre los segmentos NUEVOS (con la tolerancia de tau)
_cambio = (paso_estable != paso_estable.shift(1)) | (df["device_code"] != df["device_code"].shift(1))
_racha_id = _cambio.cumsum()
_racha_dur = df.groupby(_racha_id)["delta_t"].transform("sum")
_es_corte_real = is_gap.fillna(False) | df["delta_peso"].isna() | (paso_estable & (_racha_dur >= TAU_ELEGIDO_S))
_es_movimiento = ~_es_corte_real
_cid = _es_corte_real.groupby(df["device_code"], observed=True).cumsum()
_n_signos_por_grupo = {}
for _code in DEVICE_CODES:
    _mask = _es_movimiento & (df["device_code"] == _code)
    _grp = df.loc[_mask].groupby(_cid[_mask], observed=True)
    _n_signos_por_grupo[_code] = _grp["delta_peso"].apply(lambda s: (np.sign(s).diff().fillna(0) != 0).sum())

def _buscar_n_signos(row):
    _cid_val = _cid.loc[row["idx_inicio"]]
    return _n_signos_por_grupo[row["device_code"]].get(_cid_val, 0)

candidatos_df["n_cambios_signo"] = candidatos_df.apply(_buscar_n_signos, axis=1)
print(f"Candidatos finales (tau={TAU_ELEGIDO_S}s): {len(candidatos_df):,}")
print(candidatos_df.groupby("device_code", observed=True).size())


## Clustering (mismas features de `05_clustering_no_supervisado.ipynb`, para comparar manzanas con manzanas)


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score

FEATURES_BASE = ["duracion_s", "delta_neto_real", "max_abs_delta_g", "n_lecturas", "n_cambios_signo"]

def construir_matriz(d):
    X = d[FEATURES_BASE].copy()
    X["duracion_s"] = np.log1p(X["duracion_s"])
    X["max_abs_delta_g"] = np.log1p(X["max_abs_delta_g"])
    return StandardScaler().fit_transform(X)

for _col in ["cluster_kmeans", "cluster_agg", "cluster_gmm", "cluster_dbscan"]:
    candidatos_df[_col] = -99

for _code in DEVICE_CODES:
    _sub = candidatos_df[candidatos_df["device_code"] == _code]
    if len(_sub) < 10:
        print(f"--- {_code}: muy pocos candidatos ({len(_sub)}), se salta ---")
        continue
    _X = construir_matriz(_sub)
    _km = KMeans(n_clusters=3, random_state=0, n_init=10).fit(_X)
    _agg = AgglomerativeClustering(n_clusters=3).fit(_X)
    _gm = GaussianMixture(n_components=3, random_state=0).fit(_X)
    _gm_labels = _gm.predict(_X)
    _db = DBSCAN(eps=1.0, min_samples=5).fit(_X)
    candidatos_df.loc[_sub.index, "cluster_kmeans"] = _km.labels_
    candidatos_df.loc[_sub.index, "cluster_agg"] = _agg.labels_
    candidatos_df.loc[_sub.index, "cluster_gmm"] = _gm_labels
    candidatos_df.loc[_sub.index, "cluster_dbscan"] = _db.labels_
    print(f"--- {_code} (n={len(_sub)}) ---")
    for _nombre, _labels in [("KMeans", _km.labels_), ("Agglomerative", _agg.labels_), ("GMM", _gm_labels)]:
        print(f"  {_nombre}: tamanos={np.bincount(_labels).tolist()}, "
              f"silhouette={silhouette_score(_X, _labels):.3f}")


## Cruzar contra las etiquetas manuales ya hechas


In [ ]:
LABELS_CSV = NOTEBOOK_DIR / "data" / "candidatos_etiquetas_manuales.csv"
if LABELS_CSV.exists():
    _etq = pd.read_csv(LABELS_CSV)
    _etq["ts_inicio"] = pd.to_datetime(_etq["ts_inicio"], format="ISO8601", utc=True)
    _cruce = _etq.merge(
        candidatos_df[["ts_inicio", "duracion_s", "cluster_kmeans"]],
        on="ts_inicio", how="left",
    )
    print(_cruce)
else:
    print("Todavia no hay etiquetas manuales guardadas.")


## Exportar (mismo esquema, para la misma app)


In [ ]:
COLUMNAS_EXPORT = [
    "candidato_id", "device_code", "idx_inicio", "idx_fin", "ts_inicio", "ts_fin",
    "duracion_s", "n_lecturas", "delta_neto_g", "delta_neto_real", "max_abs_delta_g",
    "n_cambios_signo", "nivel_antes", "nivel_despues",
    "cluster_kmeans", "cluster_agg", "cluster_gmm", "cluster_dbscan",
]
CLUSTERS_CSV = NOTEBOOK_DIR / "data" / "candidatos_clusters_duracion.csv"
candidatos_df[COLUMNAS_EXPORT].sort_values(["device_code", "ts_inicio"]).to_csv(CLUSTERS_CSV, index=False)
print(f"Exportado: {CLUSTERS_CSV} ({len(candidatos_df):,} candidatos)")


## Cierre

Pendiente de completar con el resultado real una vez corrido.
